# Scaffold-Based Train, Validation and Test Split

This notebook creates scaffold-based training, validation and test sets using Bemis–Murcko molecular scaffolds. Molecules sharing the same scaffold are kept within the same split to reduce structural leakage, while scaffold-level activity groups are used to maintain a similar class distribution across the three sets. The final split contains separate training, validation and held-out scaffold test sets with zero scaffold overlap.

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split

PROCESSED_FILE = "processed_molecules.csv"
RANDOM_SEED = 42

data_df = pd.read_csv(PROCESSED_FILE)

print("Dataset loaded successfully.")
print("Dataset shape:", data_df.shape)

data_df.head()

In [ ]:
# Generating a Murcko scaffold for each molecule

def get_scaffold(smiles):

    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        return None

    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=molecule
    )

    # Some molecules do not contain rings, giving each acyclic molecule its own scaffold group.
    if scaffold == "":
        scaffold = "ACYCLIC_" + smiles

    return scaffold


data_df["scaffold"] = data_df["canonical_smiles"].apply(
    get_scaffold
)

print("Missing scaffolds:", data_df["scaffold"].isna().sum())
print("Number of unique scaffolds:", data_df["scaffold"].nunique())

data_df[
    ["canonical_smiles", "scaffold"]
].head()

In [ ]:
# molecules belonging to each scaffold

scaffold_summary = (
    data_df
    .groupby("scaffold")
    .agg(
        molecule_count=("canonical_smiles", "count"),
        dual_candidate_count=("dual_candidate", "sum")
    )
    .reset_index()
)

# Calculate the proportion of dual candidates in each scaffold
scaffold_summary["dual_fraction"] = (
    scaffold_summary["dual_candidate_count"] /
    scaffold_summary["molecule_count"]
)

# Place scaffolds into activity groups for balanced splitting
scaffold_summary["activity_group"] = pd.cut(
    scaffold_summary["dual_fraction"],
    bins=[-0.001, 0, 0.25, 0.50, 0.75, 1.001],
    labels=[
        "No dual candidates",
        "Low",
        "Medium",
        "High",
        "Mostly dual candidates"
    ],
    include_lowest=True
)

print("Total scaffolds:", len(scaffold_summary))

print("\nScaffolds in each activity group:")
print(scaffold_summary["activity_group"].value_counts())

scaffold_summary.head()

In [ ]:
# Split scaffolds into train, validation and test 

# First split: 70% training 30% temporary set

train_scaffolds, temporary_scaffolds = train_test_split(
    scaffold_summary,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=scaffold_summary["activity_group"]
)

# Second split: Divide the temporary set equally into validation and test

validation_scaffolds, test_scaffolds = train_test_split(
    temporary_scaffolds,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temporary_scaffolds["activity_group"]
)

# Create scaffold-to-split mapping
split_mapping = {}

for scaffold in train_scaffolds["scaffold"]:
    split_mapping[scaffold] = "train"

for scaffold in validation_scaffolds["scaffold"]:
    split_mapping[scaffold] = "validation"

for scaffold in test_scaffolds["scaffold"]:
    split_mapping[scaffold] = "test"

# Assign every molecule to its scaffold's split
data_df["split"] = data_df["scaffold"].map(
    split_mapping
)

print("Split assignment completed.")
print(data_df["split"].value_counts())

In [ ]:
# Checking the scaffold split  

split_summary = (
    data_df
    .groupby("split")
    .agg(
        molecule_count=("canonical_smiles", "count"),
        unique_scaffolds=("scaffold", "nunique"),
        dual_candidates=("dual_candidate", "sum"),
        dual_percentage=("dual_candidate", "mean")
    )
    .reindex(["train", "validation", "test"])
)

split_summary["dual_percentage"] = (
    split_summary["dual_percentage"] * 100
).round(2)

display(split_summary)


# Checking that no scaffold appears in more than one split
train_set = set(
    data_df.loc[data_df["split"] == "train", "scaffold"]
)

validation_set = set(
    data_df.loc[data_df["split"] == "validation", "scaffold"]
)

test_set = set(
    data_df.loc[data_df["split"] == "test", "scaffold"]
)

print("Train-validation overlap:",
      len(train_set.intersection(validation_set)))

print("Train-test overlap:",
      len(train_set.intersection(test_set)))

print("Validation-test overlap:",
      len(validation_set.intersection(test_set)))


split_assignments = data_df[
    ["canonical_smiles", "scaffold", "split"]
].copy()

split_assignments.to_csv(
    "split_assignments.csv",
    index=False
)

print("\nSaved file: split_assignments.csv")